# Cohort 6: Remove Data Skew

### m2.2xlarge EMR Cluster

In [1]:
%%configure -f
{
    "conf": {
        "spark.executor.memory": "24g", 
        "spark.executor.memoryOverhead": "6g",  
        "spark.executor.cores": "4", 
        "spark.executor.instances": "100",
        "spark.dynamicAllocation.enabled": "true",  
        "spark.dynamicAllocation.minExecutors": "25",
        "spark.dynamicAllocation.maxExecutors": "100",
        "spark.sql.shuffle.partitions": "1000",
        "spark.sql.adaptive.enabled": "true",   
        "spark.driver.memory": "24g",
        "spark.driver.memoryOverhead": "6g",
        "spark.yarn.am.memory": "4g",                
        "spark.task.cpus": "1",                     
        "spark.network.timeout": "1200s",
        "spark.rpc.askTimeout": "1200s",
        "spark.jars.packages.resolve.transitive": "true",
        "spark.executor.extraJavaOptions": "--add-exports java.base/sun.net.util=ALL-UNNAMED",
        "spark.driver.extraJavaOptions": "--add-exports java.base/sun.net.util=ALL-UNNAMED",
        "spark.eventLog.rolling.enabled": "true",
        "spark.eventLog.rolling.maxFileSize": "128m"

    }
}

In [2]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when
from pyspark.ml.linalg import Vectors
from pyspark.sql.types import StructType, StructField, DoubleType, StringType

VBox()

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
4,application_1737170788454_0005,pyspark,idle,Link,Link,None,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [3]:
# Adding a parameter tag
cohort = 'cohort6'

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [4]:
# S3 Paths
s3_bucket = f"s3://pgx-repository/ade-risk-model/Step5_Time_to_Event_Model/2_processed_datasets/{cohort}"
train_input_path = f"{s3_bucket}/train"
test_input_path = f"{s3_bucket}/test"

# Read processed train and test datasets from S3
print("Reading train and test datasets...")
train_df = spark.read.parquet(train_input_path)
test_df = spark.read.parquet(test_input_path)

print("Train and test datasets successfully loaded.")

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Reading train and test datasets...
Train and test datasets successfully loaded.

In [5]:
# Verify output
print("Train Dataframe Schema:")
train_df.printSchema()
print("Test Dataframe Schema:")
test_df.printSchema()

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Train Dataframe Schema:
root
 |-- mi_person_key: string (nullable = true)
 |-- member_age_dos: integer (nullable = true)
 |-- drug_date: date (nullable = true)
 |-- ADE_Date: date (nullable = true)
 |-- standardized_drug_name: string (nullable = true)
 |-- label: integer (nullable = true)
 |-- person_key_index: double (nullable = true)
 |-- drug_name_index: double (nullable = true)
 |-- drug_name_one_hot: vector (nullable = true)
 |-- features: vector (nullable = true)

Test Dataframe Schema:
root
 |-- mi_person_key: string (nullable = true)
 |-- member_age_dos: integer (nullable = true)
 |-- drug_date: date (nullable = true)
 |-- ADE_Date: date (nullable = true)
 |-- standardized_drug_name: string (nullable = true)
 |-- label: integer (nullable = true)
 |-- person_key_index: double (nullable = true)
 |-- drug_name_index: double (nullable = true)
 |-- drug_name_one_hot: vector (nullable = true)
 |-- features: vector (nullable = true)

# Add Polypharmacy and 'Medical Activity' Features

In [6]:
from pyspark.sql import functions as F

# Compute Features
polypharmacy_train_df = train_df.groupBy("mi_person_key").agg(
    F.countDistinct("standardized_drug_name").alias("polypharmacy")
)

activity_count_train_df = train_df.groupBy("mi_person_key").agg(
    F.count("*").alias("activity_count")
)

# Add computed features to the main DataFrame
enhanced_train_df = train_df.join(polypharmacy_train_df, on="mi_person_key", how="left") \
                .join(activity_count_train_df, on="mi_person_key", how="left").distinct()

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [7]:
from pyspark.sql.functions import histogram_numeric
from pyspark.sql.functions import lit

# Polypharmacy histogram with 20 bins
polypharm_hist = enhanced_train_df.select(histogram_numeric('polypharmacy', lit(20))).collect()

# Activity histogram with 10 bins
activity_hist = enhanced_train_df.select(histogram_numeric('activity_count', lit(10))).collect()

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

# Polypharmacy Histogram

In [8]:
polypharm_hist

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

[Row(histogram_numeric(polypharmacy, 20)=[Row(x=3, y=60530.0), Row(x=6, y=38479030.0), Row(x=10, y=71961719.0), Row(x=15, y=104849893.0), Row(x=20, y=114183794.0), Row(x=24, y=100122792.0), Row(x=29, y=79485492.0), Row(x=35, y=76882357.0), Row(x=41, y=38630748.0), Row(x=46, y=23230621.0), Row(x=51, y=18999767.0), Row(x=56, y=12176297.0), Row(x=61, y=7307855.0), Row(x=66, y=3683769.0), Row(x=70, y=4041761.0), Row(x=76, y=2352548.0), Row(x=83, y=1686702.0), Row(x=89, y=1153978.0), Row(x=98, y=242136.0), Row(x=104, y=230494.0)])]

In [9]:
from pyspark.sql.functions import histogram_numeric, lit
import matplotlib.pyplot as plt
import numpy as np

# Extract histogram data
if polypharm_hist:
    histogram = polypharm_hist[0][0]  # Access the list of Row(x, y)
    bins = [row['x'] for row in histogram]
    frequencies = [row['y'] for row in histogram]
    histogram_tuples = [(bins[i], frequencies[i]) for i in range(len(bins))]
    polypharm_histogram_df = spark.createDataFrame(histogram_tuples, ['bin', 'frequency'])

else:
    print("No histogram data available. Check dataset or parameters.")

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [10]:
%%display 
polypharm_histogram_df

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Output()

# Medical Activity Histogram

In [11]:
activity_hist

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

[Row(histogram_numeric(activity_count, 10)=[Row(x=4606, y=645946349.0), Row(x=15414, y=14158991.0), Row(x=30308, y=21297189.0), Row(x=46009, y=9018382.0), Row(x=61663, y=4172279.0), Row(x=77655, y=2010331.0), Row(x=90711, y=1177004.0), Row(x=114227, y=1243181.0), Row(x=140009, y=558949.0), Row(x=179904, y=179628.0)])]

In [12]:
from pyspark.sql.functions import histogram_numeric, lit
import matplotlib.pyplot as plt
import numpy as np

# Extract histogram data
if activity_hist:
    act_histogram = activity_hist[0][0]  # Access the list of Row(x, y)
    act_bins = [row['x'] for row in act_histogram]
    act_frequencies = [row['y'] for row in act_histogram]
    act_histogram_tuples = [(act_bins[i], act_frequencies[i]) for i in range(len(act_bins))]
    activity_histogram_df = spark.createDataFrame(act_histogram_tuples, ['bin', 'frequency'])

else:
    print("No histogram data available. Check dataset or parameters.")

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [13]:
%%display 
activity_histogram_df

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Output()

# Remove Data Skew

In [14]:
# Repartition the Data
# Create Polypharmacy and Activity Count Bins
enhanced_train_df = enhanced_train_df.withColumn(
    "polypharmacy_bin",
    F.when((F.col("polypharmacy") >= 1) & (F.col("polypharmacy") <= 2), "1-2")
     .when((F.col("polypharmacy") > 2) & (F.col("polypharmacy") <= 5), "2-5")
     .when((F.col("polypharmacy") > 5) & (F.col("polypharmacy") <= 15), "5-15")
     .when((F.col("polypharmacy") > 15) & (F.col("polypharmacy") <= 25), "15-25")
     .when((F.col("polypharmacy") > 25) & (F.col("polypharmacy") <= 50), "25-50")
     .when((F.col("polypharmacy") > 80) & (F.col("polypharmacy") <= 120), "80-120")
     .otherwise("other")  # Handles cases outside the specified ranges
).withColumn(
    "activity_count_bin",
    F.when(F.col("activity_count") <= 500, "low")
     .when((F.col("activity_count") > 500) & (F.col("activity_count") <= 1000), "medium")
     .otherwise("high")
)

# Tag High Activity Keys
enhanced_train_df = enhanced_train_df.withColumn(
    "activity_tag",
    F.when(F.col("activity_count_bin") == "high", "high")
     .otherwise("low_mid")
)

# Add a Hash Partition Column
enhanced_train_df = enhanced_train_df.withColumn(
    "partition_key",
    F.when(
        F.col("activity_tag") == "high",
        F.abs(F.hash(F.col("mi_person_key"))) % 25  
    ).otherwise(F.lit(None))  # Keep low/mid keys unaffected
)

partitioned_train_df = enhanced_train_df.repartition(
    25,
    F.when(F.col("activity_tag") == "high", F.col("partition_key"))
     .otherwise(F.col("mi_person_key")),  # Default partitioning for low/mid keys
    "polypharmacy_bin"
)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [15]:
partitioned_train_df.show(5)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+-------------+--------------+----------+----------+----------------------+-----+----------------+---------------+------------------+--------------------+------------+--------------+----------------+------------------+------------+-------------+
|mi_person_key|member_age_dos| drug_date|  ADE_Date|standardized_drug_name|label|person_key_index|drug_name_index| drug_name_one_hot|            features|polypharmacy|activity_count|polypharmacy_bin|activity_count_bin|activity_tag|partition_key|
+-------------+--------------+----------+----------+----------------------+-----+----------------+---------------+------------------+--------------------+------------+--------------+----------------+------------------+------------+-------------+
|   1011887450|            71|2018-05-02|2018-05-08|          pitavastatin|    0|        212981.0|          410.0|(7817,[410],[1.0])|(7818,[0,411],[21...|          16|           735|           15-25|            medium|     low_mid|         NULL|
|   1015924187| 

In [16]:
from pyspark.sql.functions import histogram_numeric
from pyspark.sql.functions import lit

# Partitioned histogram with 25 bins to match 25 partitions
partitioned_train_hist = partitioned_train_df.select(histogram_numeric('partition_key', lit(25))).collect()

# Extract histogram data
if partitioned_train_hist:
    partitioned_train_histogram = partitioned_train_hist[0][0]  # Access the list of Row(x, y)
    partitioned_train_bins = [row['x'] for row in partitioned_train_histogram]
    partitioned_train_frequencies = [row['y'] for row in partitioned_train_histogram]
    partitioned_histogram_train_tuples = [(partitioned_train_bins[i], partitioned_train_frequencies[i]) for i in range(len(partitioned_train_bins))]
    partitioned_histogram_train_df = spark.createDataFrame(partitioned_histogram_train_tuples, ['bin', 'frequency'])

else:
    print("No histogram data available. Check dataset or parameters.")

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [17]:
%%display 
partitioned_histogram_train_df

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Output()

# Repeat for Test Dataset (formatting purposes only)

In [18]:
from pyspark.sql import functions as F

# Repeat for Test
polypharmacy_test_df = test_df.groupBy("mi_person_key").agg(
    F.countDistinct("standardized_drug_name").alias("polypharmacy")
)

activity_count_test_df = test_df.groupBy("mi_person_key").agg(
    F.count("*").alias("activity_count")
)

# Add computed features to the main DataFrame
enhanced_test_df = test_df.join(polypharmacy_test_df, on="mi_person_key", how="left") \
                .join(activity_count_test_df, on="mi_person_key", how="left")

# Create Polypharmacy and Activity Count Bins
enhanced_test_df = enhanced_test_df.withColumn(
    "polypharmacy_bin",
    F.when((F.col("polypharmacy") >= 1) & (F.col("polypharmacy") <= 2), "1-2")
     .when((F.col("polypharmacy") > 2) & (F.col("polypharmacy") <= 5), "2-5")
     .when((F.col("polypharmacy") > 5) & (F.col("polypharmacy") <= 15), "5-15")
     .when((F.col("polypharmacy") > 15) & (F.col("polypharmacy") <= 25), "15-25")
     .when((F.col("polypharmacy") > 25) & (F.col("polypharmacy") <= 50), "25-50")
     .when((F.col("polypharmacy") > 80) & (F.col("polypharmacy") <= 120), "80-120")
     .otherwise("other")  # Handles cases outside the specified ranges
).withColumn(
    "activity_count_bin",
    F.when(F.col("activity_count") <= 50, "low")
     .when((F.col("activity_count") > 50) & (F.col("activity_count") <= 200), "medium")
     .otherwise("high")
)

# Tag High Activity Keys
enhanced_test_df = enhanced_test_df.withColumn(
    "activity_tag",
    F.when(F.col("activity_count_bin") == "high", "high")
     .otherwise("low_mid")
)

# Add a Hash Partition Column
enhanced_test_df = enhanced_test_df.withColumn(
    "partition_key",
    F.when(
        F.col("activity_tag") == "high",
        F.abs(F.hash(F.col("mi_person_key"))) % 25  
    ).otherwise(F.lit(None))  # Keep low/mid keys unaffected
)

# Repartition the Data
partitioned_test_df = enhanced_test_df.repartition(
    25,
    F.when(F.col("activity_tag") == "high", F.col("partition_key"))
     .otherwise(F.col("mi_person_key")),  # Default partitioning for low/mid keys
    "polypharmacy_bin"
)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

# Visualize Data Distribution for Updated Test Dataset

In [19]:
from pyspark.sql.functions import histogram_numeric
from pyspark.sql.functions import lit

# Partitioned histogram with 25 bins to match partitions
partitioned_test_hist = enhanced_test_df.select(histogram_numeric('partition_key', lit(25))).collect()

# Extract histogram data
if partitioned_test_hist:
    partitioned_test_histogram = partitioned_test_hist[0][0]  # Access the list of Row(x, y)
    partitioned_test_bins = [row['x'] for row in partitioned_test_histogram]
    partitioned_test_frequencies = [row['y'] for row in partitioned_test_histogram]
    partitioned_histogram_test_tuples = [(partitioned_test_bins[i], partitioned_test_frequencies[i]) for i in range(len(partitioned_test_bins))]
    partitioned_histogram_test_df = spark.createDataFrame(partitioned_histogram_test_tuples, ['bin', 'frequency'])

else:
    print("No histogram data available. Check dataset or parameters.")

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [20]:
%%display 
partitioned_histogram_test_df

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Output()

# Save Final Enhanced/Partitioned/Sorted Train and Test Datasets

In [21]:
from pyspark.sql import functions as F

s3_bucket = f"s3://pgx-repository/ade-risk-model/Step5_Time_to_Event_Model/2_enhanced_datasets/{cohort}"
train_input_path = f"{s3_bucket}/train"
test_input_path = f"{s3_bucket}/test"

# Group by 'mi_person_key' and sort by 'date'
sorted_train_df = (
    partitioned_train_df
    .sortWithinPartitions("mi_person_key", "drug_date")  # Intra-partition sorting
)

sorted_test_df = (
    partitioned_test_df
    .sortWithinPartitions("mi_person_key", "drug_date")  # Intra-partition sorting
)

# Repartition
repartitioned_train_df = (
    sorted_train_df
    .repartition(1940, F.col("partition_key"))  # Ensure data distributed across 1940 partitions
)

repartitioned_test_df = (
    sorted_test_df
    .repartition(1940, F.col("partition_key"))  # Ensure data distributed across 1940 partitions
)

# Write to S3 
repartitioned_train_df.write.mode("overwrite").parquet(train_input_path)
repartitioned_test_df.write.mode("overwrite").parquet(test_input_path)

print(f"De-Skewed Train and Test Datasets for {cohort} saved to {s3_bucket}")

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

De-Skewed Train and Test Datasets for cohort6 saved to s3://pgx-repository/ade-risk-model/Step5_Time_to_Event_Model/2_enhanced_datasets/cohort6